# Hindsight — Per-User Memory

Follows on from [`01-quickstart.ipynb`](01-quickstart.ipynb). Part 1 put every
customer in one bank (`quickstart-demo`). That's fine for learning the basics,
but it doesn't hold up in production: nothing stops one customer's `recall`
from surfacing another customer's information.

The fix Hindsight's own
[per-user-memory](https://github.com/vectorize-io/hindsight-cookbook/blob/main/notebooks/02-per-user-memory.ipynb)
example uses: **one bank per customer.** Complete isolation, and a simpler
mental model than filtering a shared bank by customer ID on every query.

## Prerequisites

Hindsight must already be running (see the main [README](../README.md)):

```bash
docker compose -f ../docker-compose.hindsight.yml up -d
```

## Installation

In [ ]:
%pip install --quiet hindsight-client nest_asyncio

## Connect to Hindsight

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from hindsight_client import Hindsight

HINDSIGHT_API_URL = "http://localhost:8888"
HINDSIGHT_UI_URL = "http://localhost:9999"

client = Hindsight(base_url=HINDSIGHT_API_URL)

# Fresh start, in case this notebook has been run before.
for bank_id in ("support-ahmet", "support-elif"):
    try:
        client.delete_bank(bank_id)
    except Exception:
        pass

## One Bank per Customer

In [ ]:
client.create_bank(bank_id="support-ahmet", name="Ahmet Yilmaz")
client.create_bank(bank_id="support-elif", name="Elif Kaya")

client.retain(
    bank_id="support-ahmet",
    content="Ahmet Yilmaz kurumsal hesap kullaniyor, odeme yontemi banka havalesi.",
)
client.retain(
    bank_id="support-elif",
    content="Elif Kaya bireysel hesap kullaniyor, iletisimde telefonu tercih ediyor.",
)

### Proof of Isolation

`recall` always ranks and returns its best-effort matches from the bank it's
given — it doesn't return an empty list just because nothing is truly
relevant. So the real isolation check isn't "how many results came back,"
it's "does Elif's own data ever show up in Ahmet's bank." It shouldn't,
no matter how the query is worded.

In [ ]:
result = client.recall(bank_id="support-ahmet", query="Elif hakkinda ne biliyoruz?")

# The real proof isn't the result count -- recall always returns its closest
# matches, even weak ones. The proof is that none of them are Elif's data.
leaked = [r.text for r in result.results if "Elif" in r.text]

print(f"support-ahmet bank'inda {len(result.results)} sonuc bulundu (bunlar Ahmet'in kendi kayitlari, asagida gorulebilir):")
for r in result.results:
    print(f"  - {r.text}")
print(f"\nBunlarin icinde Elif'e ait veri var mi? {bool(leaked)}")

### Updating a Conversation with `document_id`

A customer rarely contacts support just once. Retaining every message as a
separate memory would leave you with a pile of fragments instead of one
coherent conversation. Pass the same `document_id` on every `retain` call for
that conversation, and Hindsight replaces the previous version instead of
adding a duplicate.

In [ ]:
CONVERSATION_ID = "ahmet-kargo-sikayeti"

client.retain(
    bank_id="support-ahmet",
    content="Musteri: Kargom hala gelmedi.\nTemsilci: Kargo numaranizi kontrol ediyorum.",
    document_id=CONVERSATION_ID,
)

# Ayni musteri birkac dakika sonra tekrar yaziyor. Ayni document_id, yeni bir
# kayit degil -- konusma guncelleniyor.
client.retain(
    bank_id="support-ahmet",
    content=(
        "Musteri: Kargom hala gelmedi.\n"
        "Temsilci: Kargo numaranizi kontrol ediyorum.\n"
        "Musteri: Tesekkurler, ne zaman gelir?\n"
        "Temsilci: Yarina kadar teslim edilecek."
    ),
    document_id=CONVERSATION_ID,
)

result = client.recall(bank_id="support-ahmet", query="Kargo ne zaman gelecek?")
print("Bulunanlar:")
for r in result.results:
    print(f"  - {r.text}")

print(f"\nTek dokuman olarak gorebilirsin: {HINDSIGHT_UI_URL}/banks/support-ahmet?view=documents")

Notice the results aren't limited to the cargo conversation -- the payment
fact from the isolation check above shows up too, ranked lower. `recall`
searches everything in the bank, not just one document. That's expected:
it's a broad retrieval step, meant to be narrowed by a specific query or
handed to `reflect` for a synthesized answer, not treated as an exact
lookup.

## Cleanup

In [ ]:
client.delete_bank("support-ahmet")
client.delete_bank("support-elif")
client.close()
print("Banklar silindi, baglanti kapatildi.")